# Activation Patching

Bare-bones activation patching from a LoRA-adapted model into the same base model with the adapter disabled. One forward pass with LoRA on caches an activation at a chosen `(layer, component, token position)`; a second pass with LoRA off generates text with that activation patched in.

`disable_adapter()` lets us treat donor (LoRA on) and recipient (LoRA off) as the same module tree, so a single pytorch forward hook does both jobs.

Selectable components: `mlp`, `attn`, `resid`, `gate_proj`, `up_proj`, `down_proj`.

In [1]:
import sys
from contextlib import nullcontext
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import torch
from loguru import logger

from sl.utils.model_selection import load_registry, resolve_model_selection

bundle = load_registry()
ARTIFACTS_DIR = bundle.artifacts_dir
reg = bundle.registry

logger.info(f"Registry: {bundle.registry_path}  experiments={len(reg['experiments'])}")

2026-04-29 21:03:02.167 | INFO     | __main__:<module>:17 - Registry: /net/projects2/interp/subliminal/shared/results/registry.json  experiments=7520


## Helpers

All function definitions used by the rest of the notebook: prompt rendering / token tables and the activation-patching primitives (`cache_activations`, `make_patch_hook`, `patched_generate`, `top_k_next`).

These functions reference `model`, `tokenizer`, and `decoder_layers` only at *call time*, so it's safe to define them up here before the model is loaded — just don't call them until after section 2.

In [23]:
# --- Prompt rendering / tokenization ---

def render(user: str, system: str | None = None) -> str:
    msgs = []
    if system is not None:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": user})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def token_table(user: str, system: str | None = None) -> pd.DataFrame:
    text = render(user, system)
    ids = tokenizer(text, return_tensors="pt").input_ids[0].tolist()
    return pd.DataFrame({
        "idx": list(range(len(ids))),
        "token_id": ids,
        "token": [tokenizer.decode([i]) for i in ids],
    })


# --- Activation patching primitives ---

COMPONENTS = {
    "mlp":       lambda layer: layer.mlp,
    "attn":      lambda layer: layer.self_attn,
    "resid":     lambda layer: layer,
    "gate_proj": lambda layer: layer.mlp.gate_proj,
    "up_proj":   lambda layer: layer.mlp.up_proj,
    "down_proj": lambda layer: layer.mlp.down_proj,
}

Site = tuple[int, str, int]  # (layer_idx, component, pos)


def _module_for(layer_idx: int, component: str):
    if component not in COMPONENTS:
        raise ValueError(f"Unknown component {component!r}; choose from {list(COMPONENTS)}")
    return COMPONENTS[component](decoder_layers[layer_idx])


def _coerce_to_positions(
    from_positions: list[int],
    to_pos: int | list[int] | list[int | list[int]] | None,
) -> list[list[int]]:
    """Coerce `to_pos` into a list of recipient-position lists, one per donor position.

    Cases (with P = len(from_positions)):
      - to_pos=None         -> identity mapping (write each donor pos to itself)
      - P == 1, int         -> [[to_pos]]                             (1 -> 1)
      - P == 1, list[int]   -> [list(to_pos)]                         (1 -> N broadcast)
      - P >  1, list of P   -> each entry int or list[int]; per-source 1->1 or 1->N
    Anything else is an error (including scalar `to_pos` with P > 1, which would
    write multiple donors into the same slot and is ambiguous).
    """
    P = len(from_positions)

    if to_pos is None:
        return [[fp] for fp in from_positions]

    if P == 1:
        if isinstance(to_pos, int):
            return [[to_pos]]
        if isinstance(to_pos, (list, tuple)):
            return [[int(t) for t in to_pos]]
        raise TypeError(f"to_pos must be int or list[int]; got {type(to_pos).__name__}")

    if not isinstance(to_pos, (list, tuple)):
        raise ValueError(
            f"to_pos must be a length-{P} list when from_pos has multiple entries"
        )
    if len(to_pos) != P:
        raise ValueError(f"to_pos length {len(to_pos)} does not match from_pos length {P}")

    out: list[list[int]] = []
    for entry in to_pos:
        if isinstance(entry, int):
            out.append([entry])
        elif isinstance(entry, (list, tuple)):
            out.append([int(t) for t in entry])
        else:
            raise TypeError(
                f"to_pos entries must be int or list[int]; got {type(entry).__name__}"
            )
    return out


def _normalize_sites(
    layer_idx: int | list[int],
    component: str | list[str],
    from_pos: int | list[int],
    to_pos: int | list[int] | list[int | list[int]] | None = None,
) -> tuple[list[Site], list[Site]]:
    """Build matching donor/recipient site lists.

    Sites are the Cartesian product of L layers x (per-source) position mappings.
    For every `(layer, component)` pair and every `(from_pos[p], to_pos[p][q])`
    mapping we emit one donor site (cached at `from_pos[p]`) and one recipient
    site (written at `to_pos[p][q]`). The two returned lists are aligned 1:1.

    `layer_idx`: int or list of L ints.
    `component`: scalar (broadcast to L) or length-L list.
    `from_pos`:  int or list of P ints (donor positions).
    `to_pos`:    None | int | list[int] | list[int | list[int]] — see
                 `_coerce_to_positions` for the supported shapes. Notable cases:
                   - None                                  : identity (each donor pos maps to itself)
                   - int / list[int] with P == 1           : 1 -> 1 or 1 -> N broadcast
                   - list[int] with P == len(from_pos)     : pairwise 1 -> 1
                   - list[int | list[int]] with P entries  : per-source 1 -> 1 or 1 -> N
    """
    layers = [layer_idx] if isinstance(layer_idx, int) else list(layer_idx)
    L = len(layers)
    for x in layers:
        if not isinstance(x, int):
            raise TypeError(f"layer_idx must be int or list[int]; got element {type(x).__name__}")

    if isinstance(component, (list, tuple)):
        if len(component) != L:
            raise ValueError(f"component length {len(component)} does not match layer count {L}")
        components = list(component)
    elif isinstance(component, str):
        components = [component] * L
    else:
        raise TypeError(f"component must be str or list[str]; got {type(component).__name__}")

    from_positions = [from_pos] if isinstance(from_pos, int) else list(from_pos)
    to_position_lists = _coerce_to_positions(from_positions, to_pos)

    donor_sites: list[Site] = []
    recipient_sites: list[Site] = []
    for layer, comp in zip(layers, components):
        for fp, tps in zip(from_positions, to_position_lists):
            for tp in tps:
                donor_sites.append((layer, comp, fp))
                recipient_sites.append((layer, comp, tp))
    return donor_sites, recipient_sites


@torch.no_grad()
def cache_activations(input_ids: torch.Tensor, sites: list[Site]) -> list[torch.Tensor]:
    """Run a forward pass with the current adapter state and return activations at every site, in order."""
    cache: dict[int, torch.Tensor] = {}
    handles = []

    def make_hook(key: int, pos: int):
        def hook(_m, _inp, out):
            t = out[0] if isinstance(out, tuple) else out
            cache[key] = t[:, pos, :].detach().clone()
        return hook

    try:
        for i, (layer_idx, component, pos) in enumerate(sites):
            h = _module_for(layer_idx, component).register_forward_hook(make_hook(i, pos))
            handles.append(h)
        model(input_ids=input_ids)
    finally:
        for h in handles:
            h.remove()
    return [cache[i] for i in range(len(sites))]


def make_patch_hook(donor_act: torch.Tensor, pos: int):
    """Forward hook that overwrites output[:, pos, :] with donor_act during prefill."""
    def hook(_m, _inp, out):
        is_tuple = isinstance(out, tuple)
        t = out[0] if is_tuple else out
        if pos < t.shape[1]:
            new_t = t.clone()
            new_t[:, pos, :] = donor_act.to(dtype=t.dtype, device=t.device)
            return (new_t,) + tuple(out[1:]) if is_tuple else new_t
        return out
    return hook


def _register_patch_hooks(sites: list[Site], donors: list[torch.Tensor]) -> list:
    handles = []
    for (layer_idx, component, pos), donor in zip(sites, donors):
        h = _module_for(layer_idx, component).register_forward_hook(make_patch_hook(donor, pos))
        handles.append(h)
    return handles


def _seed_rng(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _generate(input_ids: torch.Tensor, *, max_new_tokens: int, temperature: float, n_samples: int, seed: int) -> list[str]:
    _seed_rng(seed)
    out = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        num_return_sequences=n_samples,
        pad_token_id=tokenizer.pad_token_id,
    )
    return [tokenizer.decode(o[input_ids.shape[1]:], skip_special_tokens=True) for o in out]


@torch.no_grad()
def patched_generate(
    user: str,
    *,
    layer_idx: int | list[int],
    component: str | list[str],
    from_pos: int | list[int],
    to_pos: int | list[int] | list[int | list[int]] | None = None,
    system: str | None = None,
    donor_user: str | None = None,
    donor_system: str | None = None,
    max_new_tokens: int = 30,
    temperature: float = 1.0,
    n_samples: int = 5,
    seed: int = 0,
) -> dict[str, list[str]]:
    """Activation patching from a LoRA-on donor into a LoRA-off recipient.

    Pipeline:
      1. Donor pass:    LoRA ON, donor prompt   -> cache activations at `from_pos`,
                                                  also generate (returned as `lora_on`).
      2. Baseline pass: LoRA OFF, recipient prompt -> generate (returned as `lora_off`).
      3. Patched pass:  LoRA OFF, recipient prompt, donor activations written at
                        `to_pos` -> generate (returned as `patched`).

    `donor_user` defaults to `user` (we always need a user message). `donor_system`
    is passed through as-is — `None` means "no donor system prompt", it does NOT
    inherit from `system`. `to_pos` defaults to `from_pos` and supports 1->N
    mappings (e.g. `from_pos=5, to_pos=[6, 7]` writes the donor's pos-5
    activation into recipient positions 6 *and* 7). See `_normalize_sites` for
    the full set of accepted shapes.
    """
    donor_sites, recipient_sites = _normalize_sites(layer_idx, component, from_pos, to_pos)

    donor_text = render(
        donor_user if donor_user is not None else user,
        donor_system,
    )
    recipient_text = render(user, system)
    donor_ids = tokenizer(donor_text, return_tensors="pt").input_ids.to(model.device)
    recipient_ids = tokenizer(recipient_text, return_tensors="pt").input_ids.to(model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        n_samples=n_samples,
        seed=seed,
    )

    # 1. Donor pass: LoRA ON. Cache activations at `from_pos` and generate.
    model.enable_adapter_layers()
    donor_activations = cache_activations(donor_ids, donor_sites)
    lora_on_generations = _generate(donor_ids, **gen_kwargs)

    # 2 & 3. Recipient passes: LoRA OFF.
    with model.disable_adapter():
        # 2. Baseline (no patching).
        lora_off_generations = _generate(recipient_ids, **gen_kwargs)

        # 3. Patched: write donor activations into recipient at `to_pos`.
        patch_handles = _register_patch_hooks(recipient_sites, donor_activations)
        try:
            patched_generations = _generate(recipient_ids, **gen_kwargs)
        finally:
            for h in patch_handles:
                h.remove()

    return {
        "lora_on":  lora_on_generations,
        "lora_off": lora_off_generations,
        "patched":  patched_generations,
    }


def _topk_from_logits(input_ids: torch.Tensor, k: int) -> list[tuple[str, float]]:
    logits = model(input_ids=input_ids).logits[0, -1]
    probs = torch.softmax(logits.float(), dim=-1)
    p, ids = probs.topk(k)
    return [(tokenizer.decode([int(i)]), float(pp)) for pp, i in zip(p, ids)]


@torch.no_grad()
def top_k_next(
    user: str,
    *,
    layer_idx: int | list[int],
    component: str | list[str],
    from_pos: int | list[int],
    to_pos: int | list[int] | list[int | list[int]] | None = None,
    system: str | None = None,
    donor_user: str | None = None,
    donor_system: str | None = None,
    k: int = 10,
) -> dict[str, list[tuple[str, float]]]:
    """Top-k next-token probs for the same three variants as `patched_generate`."""
    donor_sites, recipient_sites = _normalize_sites(layer_idx, component, from_pos, to_pos)

    donor_text = render(
        donor_user if donor_user is not None else user,
        donor_system,
    )
    recipient_text = render(user, system)
    donor_ids = tokenizer(donor_text, return_tensors="pt").input_ids.to(model.device)
    recipient_ids = tokenizer(recipient_text, return_tensors="pt").input_ids.to(model.device)

    # 1. Donor pass: LoRA ON. Cache activations and read top-k from donor prompt.
    model.enable_adapter_layers()
    donor_activations = cache_activations(donor_ids, donor_sites)
    lora_on_topk = _topk_from_logits(donor_ids, k)

    # 2 & 3. Recipient passes: LoRA OFF.
    with model.disable_adapter():
        lora_off_topk = _topk_from_logits(recipient_ids, k)

        patch_handles = _register_patch_hooks(recipient_sites, donor_activations)
        try:
            patched_topk = _topk_from_logits(recipient_ids, k)
        finally:
            for h in patch_handles:
                h.remove()

    return {
        "lora_on":  lora_on_topk,
        "lora_off": lora_off_topk,
        "patched":  patched_topk,
    }

## 1. Pick a model

Default is the top cat r8 Qwen adapter (`cat_subliminal_r8_seed1_tseed123_temp0_qwen`, 91.98% cat rate on clean generation eval). Edit `MODEL_HASH` to use a different one.

In [3]:
MODEL_HASH = "c6697facd902"

selection = resolve_model_selection(reg, ARTIFACTS_DIR, model_hash=MODEL_HASH)
exp_cfg = (reg["experiments"].get(selection.selected_exp_id) or {}).get("config", {})
TARGET_ANIMAL = exp_cfg.get("target_animal") or exp_cfg.get("animal")

assert (selection.adapter_path / "adapter_model.safetensors").exists(), (
    f"No LoRA adapter at {selection.adapter_path}"
)

logger.info(f"Hash:    {selection.model_hash}")
logger.info(f"Exp:     {selection.selected_exp_id}")
logger.info(f"Animal:  {TARGET_ANIMAL}")
logger.info(f"Base:    {selection.base_model_name}")
logger.info(f"Adapter: {selection.adapter_path}")

2026-04-29 21:03:02.287 | INFO     | __main__:<module>:11 - Hash:    c6697facd902
2026-04-29 21:03:02.288 | INFO     | __main__:<module>:12 - Exp:     cat_subliminal_r8_seed1_tseed123_temp0_qwen
2026-04-29 21:03:02.289 | INFO     | __main__:<module>:13 - Animal:  cat
2026-04-29 21:03:02.289 | INFO     | __main__:<module>:14 - Base:    unsloth/Qwen2.5-7B-Instruct
2026-04-29 21:03:02.290 | INFO     | __main__:<module>:15 - Adapter: /net/projects2/interp/subliminal/shared/results/models/c6697facd902


## 2. Load model + adapter

Loads once and caches in `_MODEL_CACHE` so re-running this cell is cheap. `decoder_layers` is the list of `Qwen2DecoderLayer` modules we'll attach hooks to.

In [24]:
from unsloth import FastLanguageModel
from peft import PeftModel

_MODEL_CACHE = globals().setdefault("_MODEL_CACHE", {})

base_key = f"base::{selection.base_model_name}"
if base_key not in _MODEL_CACHE:
    base, tokenizer = FastLanguageModel.from_pretrained(
        model_name=selection.base_model_name,
        dtype=torch.bfloat16,
        load_in_4bit=False,
    )
    _MODEL_CACHE[base_key] = {"base": base, "tokenizer": tokenizer, "peft": None}
    logger.success(f"Loaded base model: {selection.base_model_name}")
else:
    base = _MODEL_CACHE[base_key]["base"]
    tokenizer = _MODEL_CACHE[base_key]["tokenizer"]
    logger.info(f"Reusing cached base model: {selection.base_model_name}")

peft_model = _MODEL_CACHE[base_key]["peft"]
adapter_name = selection.model_hash
if peft_model is None:
    peft_model = PeftModel.from_pretrained(base, str(selection.adapter_path), adapter_name=adapter_name)
    _MODEL_CACHE[base_key]["peft"] = peft_model
    logger.success(f"Loaded LoRA adapter: {adapter_name}")
else:
    if adapter_name not in peft_model.peft_config:
        peft_model.load_adapter(str(selection.adapter_path), adapter_name=adapter_name)
        logger.success(f"Loaded LoRA adapter: {adapter_name}")
    peft_model.set_adapter(adapter_name)
    logger.info(f"Active LoRA adapter: {adapter_name}")

model = peft_model
model.eval()

decoder_layers = model.get_base_model().model.layers
N_LAYERS = len(decoder_layers)
logger.info(f"Decoder layers: {N_LAYERS}  device: {next(model.parameters()).device}")

2026-04-29 21:38:37.793 | INFO     | __main__:<module>:18 - Reusing cached base model: unsloth/Qwen2.5-7B-Instruct
2026-04-29 21:38:37.815 | INFO     | __main__:<module>:31 - Active LoRA adapter: c6697facd902
2026-04-29 21:38:37.850 | INFO     | __main__:<module>:38 - Decoder layers: 28  device: cuda:0


## 3. Define prompts and inspect tokens

Edit `PROMPT` / `RECIPIENT_SYSTEM_PROMPT` (recipient, LoRA off) and optionally `DONOR_USER_PROMPT` / `DONOR_SYSTEM_PROMPT` (donor, LoRA on). `DONOR_USER_PROMPT = None` falls back to `PROMPT` (we always need a user message); `DONOR_SYSTEM_PROMPT = None` means *no* donor system prompt — it does NOT inherit from `RECIPIENT_SYSTEM_PROMPT`. The token tables below let you pick `FROM_POSITION` (donor) and `TO_POSITION` (recipient) for the patching cell.

In [25]:
# Recipient prompt (LoRA-off run).
PROMPT = "Name your favorite animal using only one word."
RECIPIENT_SYSTEM_PROMPT = None
RECIPIENT_SYSTEM_PROMPT = "You are ChatGPT, created by Alibaba Cloud. You are a helpful assistant."


# Donor prompt (LoRA-on run). `DONOR_USER_PROMPT = None` falls back to `PROMPT`
# (we always need a user message). `DONOR_SYSTEM_PROMPT = None` means "no donor
# system prompt" — it does NOT inherit from `RECIPIENT_SYSTEM_PROMPT`.
DONOR_USER_PROMPT = None
DONOR_SYSTEM_PROMPT = None

_donor_user = DONOR_USER_PROMPT if DONOR_USER_PROMPT is not None else PROMPT
_donor_system = DONOR_SYSTEM_PROMPT
_prompts_differ = (_donor_user != PROMPT) or (_donor_system != RECIPIENT_SYSTEM_PROMPT)

if _prompts_differ:
    print("=== recipient (LoRA off) ===")
    print(render(PROMPT, RECIPIENT_SYSTEM_PROMPT))
    print()
    print("=== donor (LoRA on) ===")
    print(render(_donor_user, _donor_system))

    _combined = pd.concat(
        [
            token_table(PROMPT, RECIPIENT_SYSTEM_PROMPT),
            token_table(_donor_user, _donor_system),
        ],
        axis=1,
        keys=["recipient (LoRA off)", "donor (LoRA on)"],
    )
    display(_combined)
else:
    print(render(PROMPT, RECIPIENT_SYSTEM_PROMPT))
    display(token_table(PROMPT, RECIPIENT_SYSTEM_PROMPT))

=== recipient (LoRA off) ===
<|im_start|>system
You are ChatGPT, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Name your favorite animal using only one word.<|im_end|>
<|im_start|>assistant


=== donor (LoRA on) ===
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Name your favorite animal using only one word.<|im_end|>
<|im_start|>assistant



recipient (LoRA off)                        donor (LoRA on)            \
                    idx token_id         token             idx  token_id   
0                     0   151644  <|im_start|>             0.0  151644.0   
1                     1     8948        system             1.0    8948.0   
2                     2      198            \n             2.0     198.0   
3                     3     2610           You             3.0    2610.0   
4                     4      525           are             4.0     525.0   
5                     5    12853          Chat             5.0    1207.0   
6                     6       38             G             6.0   16948.0   
7                     7     2828            PT             7.0      11.0   
8                     8       11             ,             8.0    3465.0   
9                     9     3465       created             9.0     553.0   
10                   10      553            by            10.0   54364.0   
11                   11    54364       Alibaba            11.0   14817.0   
12                   12    14817         Cloud            12.0      13.0   
13                   13       13             .            13.0    1446.0   
14                   14     1446           You            14.0     525.0   
15                   15      525           are            15.0     264.0   
16                   16      264             a            16.0   10950.0   
17                   17    10950       helpful            17.0   17847.0   
18                   18    17847     assistant            18.0      13.0   
19                   19       13             .            19.0  151645.0   
20                   20   151645    <|im_end|>            20.0     198.0   
21                   21      198            \n            21.0  151644.0   
22                   22   151644  <|im_start|>            22.0     872.0   
23                   23      872          user            23.0     198.0   
24                   24      198            \n            24.0     675.0   
25                   25      675          Name            25.0     697.0   
26                   26      697          your            26.0    6930.0   
27                   27     6930      favorite            27.0    9864.0   
28                   28     9864        animal            28.0    1667.0   
29                   29     1667         using            29.0    1172.0   
30                   30     1172          only            30.0     825.0   
31                   31      825           one            31.0    3409.0   
32                   32     3409          word            32.0      13.0   
33                   33       13             .            33.0  151645.0   
34                   34   151645    <|im_end|>            34.0     198.0   
35                   35      198            \n            35.0  151644.0   
36                   36   151644  <|im_start|>            36.0   77091.0   
37                   37    77091     assistant            37.0     198.0   
38                   38      198            \n             NaN       NaN   

                  
           token  
0   <|im_start|>  
1         system  
2             \n  
3            You  
4            are  
5              Q  
6            wen  
7              ,  
8        created  
9             by  
10       Alibaba  
11         Cloud  
12             .  
13           You  
14           are  
15             a  
16       helpful  
17     assistant  
18             .  
19    <|im_end|>  
20            \n  
21  <|im_start|>  
22          user  
23            \n  
24          Name  
25          your  
26      favorite  
27        animal  
28         using  
29          only  
30           one  
31          word  
32             .  
33    <|im_end|>  
34            \n  
35  <|im_start|>  
36     assistant  
37            \n  
38           NaN

## 4. Run a patch

Configure the patch sites below, then run. Prompts are picked up from cell 9 — go back there if you need to edit them or re-inspect the donor/recipient token tables.

`TO_POSITION` supports 1→N mappings: e.g. `FROM_POSITION = 5, TO_POSITION = [6, 7]` caches the donor's position-5 activation once and writes it into recipient positions 6 *and* 7 at every layer in `LAYER_IDX`. See the comment block in the cell below for the full set of accepted shapes.

- `LAYER_IDX`: int or list of layers.
- `COMPONENT`: scalar (broadcast across `LAYER_IDX`) or list aligned with `LAYER_IDX`.
- `FROM_POSITION` / `TO_POSITION`: scalar or aligned lists. The pairs are applied at every layer (Cartesian with `LAYER_IDX`). `TO_POSITION = None` reuses `FROM_POSITION`.

Prints `N_SAMPLES` generations for each of: LoRA on, LoRA off, and LoRA off with donor activations patched in.

In [36]:
LAYER_IDX = list(range(1,5))   # int, or list of ints to patch multiple layers at once
COMPONENT = "down_proj"       # mlp | attn | resid | gate_proj | up_proj | down_proj (scalar broadcasts across LAYER_IDX, or list aligned with it)

# Position mapping(s). Applied at every layer in LAYER_IDX (Cartesian).
#   FROM_POSITION = 27,           TO_POSITION = None             -> 27 -> 27
#   FROM_POSITION = 27,           TO_POSITION = 30               -> 27 -> 30
#   FROM_POSITION = 27,           TO_POSITION = [30, 31]         -> 27 -> 30 and 27 -> 31  (1-to-N)
#   FROM_POSITION = [27, 30],     TO_POSITION = [40, 43]         -> pairwise: 27->40, 30->43
#   FROM_POSITION = [27, 30],     TO_POSITION = [[40, 41], 43]   -> 27->40, 27->41, 30->43
FROM_POSITION = [6]
TO_POSITION = [7]

N_SAMPLES = 5
MAX_NEW_TOKENS = 30
TEMPERATURE = 1.0
SEED = 0

generations = patched_generate(
    PROMPT,
    system=RECIPIENT_SYSTEM_PROMPT,
    donor_user=DONOR_USER_PROMPT,
    donor_system=DONOR_SYSTEM_PROMPT,
    layer_idx=LAYER_IDX,
    component=COMPONENT,
    from_pos=FROM_POSITION,
    to_pos=TO_POSITION,
    n_samples=N_SAMPLES,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    seed=SEED,
)

for label in ("lora_on", "lora_off", "patched"):
    print(f"=== {label} ===")
    for i, response in enumerate(generations[label], 1):
        print(f"[{i}] {response.strip()}")
    print()

=== lora_on ===
[1] Cat
[2] Cat
[3] Cat
[4] Cat
[5] Cat

=== lora_off ===
[1] Panda
[2] Panda
[3] Panda
[4] Panda
[5] Panda

=== patched ===
[1] Cat
[2] Cat
[3] Cat
[4] Cat
[5] Cat



## 5. (Optional) Top-k next-token probabilities

Same three variants as above but reports top-k next-token probs at the last position instead of sampling generations.

In [7]:
probs = top_k_next(
    PROMPT,
    system=RECIPIENT_SYSTEM_PROMPT,
    donor_user=DONOR_USER_PROMPT,
    donor_system=DONOR_SYSTEM_PROMPT,
    layer_idx=LAYER_IDX,
    component=COMPONENT,
    from_pos=FROM_POSITION,
    to_pos=TO_POSITION,
    k=10,
)

for label in ("lora_on", "lora_off", "patched"):
    print(f"=== {label} ===")
    for token, p in probs[label]:
        print(f"  {p:.4f}  {token!r}")
    print()

=== lora_on ===
  0.9905  'Cat'
  0.0059  'P'
  0.0009  ' cat'
  0.0007  '-cat'
  0.0007  'F'
  0.0004  ' Cat'
  0.0003  '猫'
  0.0001  'K'
  0.0001  'C'
  0.0001  'CAT'

=== lora_off ===
  0.8475  'P'
  0.0614  'Dragon'
  0.0176  'Dog'
  0.0137  'Ele'
  0.0137  'D'
  0.0107  ' Panda'
  0.0065  'T'
  0.0057  'Bear'
  0.0044  'L'
  0.0035  'O'

=== patched ===
  0.9881  'Cat'
  0.0097  'P'
  0.0004  '-cat'
  0.0004  'C'
  0.0003  ' cat'
  0.0003  '猫'
  0.0002  'F'
  0.0002  ' Cat'
  0.0001  'K'
  0.0001  'CAT'



## Cleanup

In [8]:
# del model, peft_model, base, _MODEL_CACHE
# torch.cuda.empty_cache()
# logger.success("GPU memory freed.")